# Tutorial 1.6: Framework Integrations

![](images/7_Framework-Integrations.png)

## Working with Multiple GenAI Frameworks

Welcome to framework integrations! MLflow supports 30+ GenAI frameworks, making it the most flexible platform for LLM development. This notebook shows you how to work with the most popular frameworks.

### What You'll Learn
- Overview of MLflow's framework integrations
- Working with LangChain (chains and agents)
- Working with LlamaIndex (document indexing and RAG)
- Working with LangGraph (stateful graph workflows)
- Comparing frameworks and choosing the right one
- Best practices for each framework

### Prerequisites
- Completed previous notebooks (1.1-1.5)
- OpenAI API key or Databricks AI Gateway configured

### Estimated Time: 15-20 minutes

---
## Step 1: Framework Overview

### MLflow's 40+ [Integrations](https://mlflow.org/docs/latest/genai/tracing/integrations/)

MLflow provides automatic tracing for:

**LLM Providers:**
- OpenAI, Anthropic, Cohere, Azure OpenAI
- AWS Bedrock, Google Vertex AI
- Ollama, vLLM, Together AI

**Frameworks:**
- LangChain, LangGraph, LlamaIndex, Haystack
- DSPy, AutoGen, CrewAI
- Guardrails AI, Phoenix

### Comparison Matrix

| Feature | OpenAI | LangChain | LangGraph | LlamaIndex |
|---------|--------|-----------|-----------|------------|
| **Complexity** | Low | Medium | Medium-High | Medium |
| **Learning Curve** | Easy | Medium | Medium | Medium |
| **Use Case** | Direct calls | Workflows | Agents/routing | Doc Q&A |
| **Tracing** | ✅ Auto | ✅ Auto | ✅ Auto | ✅ Auto |
| **Agents** | Manual | ✅ Built-in | ✅ Built-in | ✅ Built-in |
| **RAG** | Manual | ✅ Built-in | ✅ Built-in | ✅ Built-in |
| **Stateful graphs** | ❌ | ❌ | ✅ Built-in | ❌ |
| **Customization** | Full | High | Full | High |
| **Performance** | Fast | Medium | Medium | Medium |

### When to Use Each

**OpenAI Direct API:**
- Simple Q&A applications
- Maximum control over prompts
- Lowest latency
- Custom implementations

**LangChain:**
- Complex multi-step workflows
- Agent applications with tools
- Need for abstractions
- Rapid prototyping

**LangGraph:**
- Workflows that branch based on LLM output
- Iterative refinement (loops/cycles)
- Multi-agent orchestration
- Any workflow that benefits from explicit state management

**LlamaIndex:**
- Document-heavy applications
- Advanced indexing strategies
- Multiple data sources
- Knowledge management

---
## Step 2: Environment Setup

In [ ]:
# Install additional frameworks
!uv add langchain langchain-openai

print("✅ Frameworks installed")

In [ ]:
import mlflow
from dotenv import load_dotenv
from utils.clnt_utils import (
    is_databricks_client, 
    is_databricks_ai_gateway_client,
    get_databricks_ai_gateway_client,
    get_openai_client,
    get_ai_gateway_model_names
)

# Load environment
load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri("http://localhost:5000")

# Determine provider and initialize client
use_databricks_provider = is_databricks_client()
use_databricks_ai_gateway = is_databricks_ai_gateway_client()

if use_databricks_ai_gateway:
    client = get_databricks_ai_gateway_client()
    model_name = get_ai_gateway_model_names()[0]
    provider_name = "Databricks AI Gateway"
else:
    client = get_openai_client()
    model_name = "gpt-5-mini"
    provider_name = "OpenAI"

print(f"✅ Environment configured: using {provider_name} client")
print(f"   MLflow version: {mlflow.__version__}")
print(f"   Tracking URI: {mlflow.get_tracking_uri()}")
print(f"   Model name: {model_name}")

# Enable OpenAI autologging
mlflow.openai.autolog()

mlflow.set_experiment("09-framework-integrations")

print("✅ OpenAI autologging: ENABLED")

---
## Step 3: LangChain Framework

LangChain provides abstractions for building LLM applications.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Enable LangChain autologging
mlflow.langchain.autolog()

print("✅ LangChain autologging enabled")

In [ ]:
from utils.clnt_utils import (
    get_databricks_langchain_chat_client, 
    get_langchain_chat_openai_client,
    get_databricks_ai_gateway_langchain_client
)

# Simple LangChain chain
print("\n🔗 LangChain Example 1: Simple Chain\n")

# Create prompt template
prompt = ChatPromptTemplate.from_template(
    "You are a {role}. Answer: {question}"
)

# Create LLM based on provider
if use_databricks_ai_gateway:
    llm = get_databricks_ai_gateway_langchain_client(model_name, temperature=1.0)
elif use_databricks_provider:
    llm = get_databricks_langchain_chat_client(model_name, temperature=1.0)
else:
    llm = get_langchain_chat_openai_client(model_name, temperature=1.0)

# Create chain using LCEL (LangChain Expression Language)
chain = prompt | llm | StrOutputParser()

# Run chain (automatically traced!)
result = chain.invoke({
    "role": "MLflow expert",
    "question": "What makes LangChain different from using OpenAI directly?"
})

print(result)
print("\n✅ Chain execution fully traced!")
print("   - Prompt construction")
print("   - LLM call")
print("   - Output parsing")

### # More complex chain with multiple steps


In [ ]:
# More complex chain with multiple steps
print("\n🔗 LangChain Example 2: Multi-Step Chain\n")


# Step 1: Generate topic
topic_prompt = ChatPromptTemplate.from_template(
    "Generate a technical topic about {domain}"
)
topic_chain = topic_prompt | llm | StrOutputParser()

# Step 2: Create outline
outline_prompt = ChatPromptTemplate.from_template(
    "Create a 3-point outline for: {topic}"
)
outline_chain = outline_prompt | llm | StrOutputParser()

# Execute pipeline
topic = topic_chain.invoke({"domain": "MLOps"})
print(f"Topic: {topic}\n")

outline = outline_chain.invoke({"topic": topic})
print(f"Outline:\n{outline}")

print("\n✅ Multi-step chain traced!")
print("   Each chain creates separate spans")
print("   Full execution visible in MLflow UI")

### LangChain Strengths

- ✅ **Abstractions**: Reusable components
- ✅ **Chains**: Complex multi-step workflows
- ✅ **Agents**: Built-in agent patterns
- ✅ **Tools**: Easy tool integration
- ✅ **Community**: Large ecosystem

---
## Step 4: LlamaIndex Framework

LlamaIndex specializes in document indexing and retrieval-augmented generation (RAG).

In [ ]:
import os
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.openai import OpenAI as LlamaIndexOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Enable LlamaIndex autologging
mlflow.llama_index.autolog()

# Configure LlamaIndex Settings based on provider
# LlamaIndex defaults to OpenAI, so we need to explicitly configure it for Databricks AI Gateway
if use_databricks_ai_gateway:
    # Get Databricks AI Gateway credentials
    databricks_token = os.environ.get("DATABRICKS_TOKEN")
    ai_gateway_base_url = os.environ.get("AI_GATEWAY_BASE_URL")
    
    # Configure LLM for Databricks AI Gateway (OpenAI-compatible endpoint)
    Settings.llm = LlamaIndexOpenAI(
        model=os.environ.get("AI_GATEWAY_LLM_MODEL", "jsd-gpt-5-2"),
        api_key=databricks_token,
        api_base=ai_gateway_base_url
    )
    
    # Configure embedding model for Databricks AI Gateway
    # Use model_name (not model) to bypass OpenAIEmbeddingModelType enum validation
    Settings.embed_model = OpenAIEmbedding(
        model_name=os.environ.get("AI_GATEWAY_EMBED_MODEL", "jsd-text-embedding-3-small"),
        api_key=databricks_token,
        api_base=ai_gateway_base_url
    )
    print("✅ LlamaIndex configured for Databricks AI Gateway")
    print(f"   LLM: {Settings.llm.model}")
    print(f"   Embeddings: {Settings.embed_model.model_name}")
else:
    # Use default OpenAI settings (requires OPENAI_API_KEY)
    print("✅ LlamaIndex using default OpenAI configuration")

print("✅ LlamaIndex autologging enabled")

In [ ]:
# Create sample documents
print("\n📚 LlamaIndex Example: Document Q&A\n")

documents = [
    Document(text="MLflow is an open source platform for the complete ML lifecycle. It provides experiment tracking, model registry, and deployment capabilities."),
    Document(text="MLflow Tracing captures the complete execution of GenAI applications, including LLM calls, retrieval steps, and tool usage."),
    Document(text="MLflow integrates with 30+ frameworks including OpenAI, LangChain, LlamaIndex, and more."),
    Document(text="MLflow supports collaborative development with experiment sharing, prompt management, and model versioning."),
]

# Create index (automatically traced)
# Note: This uses the LLM and embedding model configured in Settings above
index = VectorStoreIndex.from_documents(documents)

# Create query engine
query_engine = index.as_query_engine()

# Query the index (automatically traced)
response = query_engine.query("What tracing capabilities does MLflow have?")

print("Query: What tracing capabilities does MLflow have?")
print(f"\nAnswer: {response}")

print("\n✅ LlamaIndex execution fully traced!")
print("   - Document indexing")
print("   - Query embedding")
print("   - Retrieval")
print("   - Response synthesis")

---
## Step 5: LangGraph Framework

LangGraph builds stateful, multi-node graphs where LLM output controls which path to take — ideal for branching workflows, iterative refinement, and multi-agent orchestration.

This example shows a **customer service triage bot**: an incoming message is classified, then routed to the appropriate specialist node (billing, tech support, or general inquiry).

### Customer Service Triage — Graph Flow

```
                        ┌───────────┐
                        │   START   │
                        └─────┬─────┘
                              │
                              ▼
                       ┌─────────────┐
                       │  classify   │  ← LLM picks a category
                       └──────┬──────┘
                              │
           ┌──────────────────┼──────────────────┐
           │ "billing"        │ "tech_support"   │ "customer_inquiry"
           ▼                  ▼                  ▼
  ┌────────────────┐  ┌──────────────────┐  ┌──────────────────────┐
  │ handle_billing │  │ handle_tech_     │  │ handle_customer_     │
  │                │  │ support          │  │ inquiry              │
  └───────┬────────┘  └────────┬─────────┘  └──────────┬───────────┘
          │                    │                        │
          └────────────────────┴────────────────────────┘
                                        │
                                        ▼
                                   ┌─────────┐
                                   │   END   │
                                   └─────────┘
```

Each invocation follows exactly one path: `START → classify → [handler] → END`.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# mlflow.langchain.autolog() is already enabled — covers LangGraph automatically

# ── Shared state ─────────────────────────────────────────────────────────────
class CustomerServiceState(TypedDict):
    message: str
    category: str    # filled by classify node
    response: str    # filled by handler node

# ── Node 1: classify the incoming message ────────────────────────────────────
def classify(state: CustomerServiceState) -> CustomerServiceState:
    prompt = ChatPromptTemplate.from_template(
        "You are a customer service triage agent. "
        "Classify the following message into exactly one category: "
        "'billing', 'tech_support', or 'customer_inquiry'. "
        "Reply with one word only.\n\nMessage: {message}"
    )
    chain = prompt | llm | StrOutputParser()
    category = chain.invoke({"message": state["message"]}).strip().lower()
    # Normalize to a known category
    if "billing" in category:
        category = "billing"
    elif "tech" in category:
        category = "tech_support"
    else:
        category = "customer_inquiry"
    return {"category": category}

# ── Node 2a: billing specialist ──────────────────────────────────────────────
def handle_billing(state: CustomerServiceState) -> CustomerServiceState:
    prompt = ChatPromptTemplate.from_template(
        "You are a billing specialist. Help the customer with their billing issue.\n\n"
        "Customer message: {message}"
    )
    chain = prompt | llm | StrOutputParser()
    return {"response": chain.invoke({"message": state["message"]})}

# ── Node 2b: technical support engineer ──────────────────────────────────────
def handle_tech_support(state: CustomerServiceState) -> CustomerServiceState:
    prompt = ChatPromptTemplate.from_template(
        "You are a technical support engineer. Help the customer resolve their technical issue.\n\n"
        "Customer message: {message}"
    )
    chain = prompt | llm | StrOutputParser()
    return {"response": chain.invoke({"message": state["message"]})}

# ── Node 2c: customer success agent ──────────────────────────────────────────
def handle_customer_inquiry(state: CustomerServiceState) -> CustomerServiceState:
    prompt = ChatPromptTemplate.from_template(
        "You are a customer success agent. Answer the customer's question helpfully.\n\n"
        "Customer message: {message}"
    )
    chain = prompt | llm | StrOutputParser()
    return {"response": chain.invoke({"message": state["message"]})}

# ── Routing function ──────────────────────────────────────────────────────────
def route(state: CustomerServiceState) -> Literal["handle_billing", "handle_tech_support", "handle_customer_inquiry"]:
    routes = {
        "billing": "handle_billing",
        "tech_support": "handle_tech_support",
        "customer_inquiry": "handle_customer_inquiry",
    }
    return routes.get(state["category"], "handle_customer_inquiry")

# ── Build and compile the graph ───────────────────────────────────────────────
builder = StateGraph(CustomerServiceState)
builder.add_node("classify", classify)
builder.add_node("handle_billing", handle_billing)
builder.add_node("handle_tech_support", handle_tech_support)
builder.add_node("handle_customer_inquiry", handle_customer_inquiry)

builder.set_entry_point("classify")
builder.add_conditional_edges("classify", route)
builder.add_edge("handle_billing", END)
builder.add_edge("handle_tech_support", END)
builder.add_edge("handle_customer_inquiry", END)

app = builder.compile()

# ── Run sample messages (one per route) ───────────────────────────────────────
print("🔀 LangGraph Example: Customer Service Triage\n")

test_messages = [
    "I was charged twice for my subscription this month.",
    "The app crashes whenever I try to export a report.",
    "Can you explain the difference between the Pro and Enterprise plans?",
]

for msg in test_messages:
    result = app.invoke({"message": msg})
    print(f"Message  : {result['message']}")
    print(f"Category : {result['category']}")
    print(f"Response : {result['response'][:200]}...")
    print()

print("✅ LangGraph execution fully traced!")
print("   🔍 View in MLflow UI — each invocation produces a hierarchical trace:")
print("      - Root span: full graph invocation")
print("      - 'classify' span: LLM call that picks the route")
print("      - 'handle_billing' / 'handle_tech_support' / 'handle_customer_inquiry': routed handler")

### LangGraph Strengths

- ✅ **Stateful graphs**: Shared state flows between nodes via `TypedDict`
- ✅ **Conditional routing**: Branch to different nodes based on state
- ✅ **Cycles**: Support for loops and iterative refinement
- ✅ **Visibility**: Each node appears as its own span in the MLflow trace
- ✅ **LangChain compatible**: Reuses LangChain prompts, LLMs, and parsers

In [ ]:
# Render the compiled graph — run this after the cell above has executed
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: print ASCII representation if graphviz/playwright not available
    print(app.get_graph().draw_ascii())

---
## Step 6: Best Practices

### OpenAI Best Practices

```python
# DO:
✅ Use structured prompts
✅ Implement retry logic
✅ Handle rate limits
✅ Stream responses for UX
✅ Cache results when possible

# DON'T:
❌ Hardcode prompts
❌ Ignore error responses
❌ Skip cost tracking
❌ Use synchronous calls in production
```

### LangChain Best Practices

```python
# DO:
✅ Use LCEL for chains
✅ Leverage built-in components
✅ Test chains independently
✅ Use async for better performance
✅ Enable debug mode during development

# DON'T:
❌ Over-abstract simple use cases
❌ Ignore performance overhead
❌ Skip error handling in chains
❌ Use deprecated components
```

### LangGraph Best Practices

```python
# DO:
✅ Use TypedDict for shared state — keep fields minimal
✅ Use conditional edges for branching logic
✅ Enable mlflow.langchain.autolog() to trace each node
✅ Test nodes independently before wiring the graph
✅ Use cycles sparingly — add a step counter to prevent infinite loops

# DON'T:
❌ Use LangGraph for simple linear chains (LCEL is simpler)
❌ Store large objects in state (keep it lightweight)
❌ Forget to handle the case where routing falls through
```

### LlamaIndex Best Practices

```python
# DO:
✅ Choose appropriate index type for your use case
✅ Chunk documents thoughtfully
✅ Use metadata for filtering
✅ Cache embeddings when possible
✅ Monitor retrieval quality

# DON'T:
❌ Index without preprocessing
❌ Use default settings blindly
❌ Ignore index refresh strategies
❌ Skip evaluation of retrievals
```

---
## Summary

In this notebook, you learned:

1. ✅ Overview of MLflow's 30+ framework integrations
2. ✅ Working with LangChain (chains and workflows)
3. ✅ Working with LlamaIndex (document indexing and RAG)
4. ✅ Working with LangGraph (stateful graphs with conditional routing)
5. ✅ Best practices for each framework

### Key Takeaways

- **All frameworks** are automatically traced by MLflow
- **Choose based on use case**, not hype
- **OpenAI** for simplicity and performance (used throughout the series)
- **LangChain** for complex workflows and agents
- **LangGraph** for stateful, branching, or cyclical agent workflows
- **LlamaIndex** for document-heavy applications
- **Mix frameworks** when it makes sense

### What's Next?

**📓 Notebook 1.7: Evaluating Agents**

Learn how to evaluate your GenAI applications:
- LLM-as-Judge evaluation patterns
- MLflow built-in scorers
- Custom scorers with @scorer decorator
- DeepEval integration

### Additional Resources

- [MLflow LangChain Integration](https://mlflow.org/docs/latest/llms/langchain/index.html)
- [MLflow LlamaIndex Integration](https://mlflow.org/docs/latest/llms/llama-index/index.html)
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Framework Examples](https://github.com/mlflow/mlflow/tree/master/examples)